
# Learning the committor with GNN descriptors for alanine dipeptide

This tutorial shows how to train a committor model using the Graph Neural Network (GNN) module of `mlcolvar`. We will work with the alanine dipeptide system that is also used in the [`ex_committor.ipynb`](https://github.com/Zhang-pchao/mlcolvar-gnn/blob/gnn/docs/notebooks/examples/ex_committor.ipynb) example, but here the molecular configurations are represented as graphs rather than with hand-crafted descriptors. The workflow closely follows the ideas presented in the committor tutorials and examples, while highlighting the GNN-specific components.

> **Prerequisites:** familiarity with the [`cvs_committor.ipynb`](https://github.com/Zhang-pchao/mlcolvar-gnn/blob/gnn/docs/notebooks/tutorials/cvs_committor.ipynb) tutorial and basic PyTorch Lightning usage.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zhang-pchao/mlcolvar-gnn/blob/gnn/docs/notebooks/tutorials/cvs_committor_gnn.ipynb)



## 1. Environment setup

The first code cell installs additional dependencies when running on Google Colab and imports the packages that we will use throughout the notebook. We also set the global random seed and the default floating point precision to double precision to match the rest of the `mlcolvar` documentation.


In [ ]:
# Colab setup
import os

if os.getenv('COLAB_RELEASE_TAG'):
    import subprocess
    subprocess.run('wget https://raw.githubusercontent.com/Zhang-pchao/mlcolvar-gnn/gnn/colab_setup.sh', shell=True, check=True)
    cmd = subprocess.run('bash colab_setup.sh GNN', shell=True, check=True, stdout=subprocess.PIPE)
    print(cmd.stdout.decode('utf-8'))

# Imports
import torch
import lightning
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from typing import List, Dict

from mlcolvar.utils.io import load_dataframe
from mlcolvar.graph import data as gdata
from mlcolvar.graph.cvs.committor import GraphCommittor
from mlcolvar.graph.cvs.committor import utils as gnn_utils

# Make plots look nice
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 110

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
torch.set_default_dtype(torch.float64)



## 2. System information and helper utilities

Compared to the fully-connected neural network example, the GNN implementation needs explicit knowledge of the atomic species because the node attributes of the graph are built from the atomic numbers. The alanine dipeptide dataset contains 10 heavy atoms (carbons, nitrogens and oxygens). The COLVAR files store fractional coordinates (`p1.a`, `p1.b`, `p1.c`, …) and the triclinic cell vectors (`cell.ax`, `cell.ay`, …) in nanometers. We will convert them to Cartesian positions in ångström before constructing the graphs.

The next cell defines a few reusable helper functions:

* `load_committor_dataframe` downloads the COLVAR files and concatenates them into a single Pandas dataframe while keeping track of the basin labels.
* `fractional_to_cartesian` converts the scaled coordinates to Cartesian positions using the cell matrix.
* `dataframe_to_graph_dataset` builds a `GraphDataSet` from the dataframe by instantiating the `Configuration` objects expected by the GNN data pipeline. The helper also allows us to attach the bias values that are required to compute the training weights.


In [ ]:
# Mapping from atom type index (used in the original tutorial) to atomic numbers
ATOM_TYPE_TO_Z = {0: 6,  # carbon
                  1: 8,  # oxygen
                  2: 7}  # nitrogen

# Atom-type sequence for the ten heavy atoms of alanine dipeptide
ALANINE_ATOM_TYPES = [0, 0, 1, 2, 0, 0, 0, 1, 2, 0]
ALANINE_ATOMIC_NUMBERS = [ATOM_TYPE_TO_Z[i] for i in ALANINE_ATOM_TYPES]


def load_committor_dataframe(filenames: List[str],
                              load_args: List[Dict],
                              label_offset: int = 0) -> pd.DataFrame:
    '''Load multiple COLVAR files into a single dataframe.

    Each file receives an integer label based on its order in the list.
    '''
    if len(filenames) != len(load_args):
        raise ValueError('`filenames` and `load_args` must have the same length.')

    frames = []
    for idx, (fname, args) in enumerate(zip(filenames, load_args)):
        df = load_dataframe(fname, **args, delete_download=True)
        df = df.copy()
        df['label'] = label_offset + idx
        frames.append(df)

    dataframe = pd.concat(frames, ignore_index=True)
    dataframe = dataframe.fillna({'opes.bias': 0.0, 'bias': 0.0})
    return dataframe


def fractional_to_cartesian(fractional: np.ndarray, cell: np.ndarray) -> tuple:
    '''Convert fractional coordinates to Cartesian positions (in ångström).'''
    cell_angstrom = cell * 10.0
    cartesian_nm = fractional @ cell
    cartesian_angstrom = cartesian_nm * 10.0
    return cartesian_angstrom, cell_angstrom


def dataframe_to_graph_dataset(dataframe: pd.DataFrame,
                               cutoff_angstrom: float,
                               label_column: str = 'label',
                               weight_column: str = None,
                               show_progress: bool = True) -> tuple:
    '''Convert a dataframe of COLVAR records into a GraphDataSet.'''
    configurations = []
    n_atoms = len(ALANINE_ATOMIC_NUMBERS)

    position_columns = [[f'p{i}.{axis}' for axis in 'abc'] for i in range(1, n_atoms + 1)]
    cell_columns = ['cell.ax', 'cell.ay', 'cell.az',
                    'cell.bx', 'cell.by', 'cell.bz',
                    'cell.cx', 'cell.cy', 'cell.cz']

    for row in dataframe.itertuples(index=False):
        fractional = np.array([[getattr(row, col) for col in cols] for cols in position_columns])
        cell = np.array([getattr(row, c) for c in cell_columns]).reshape(3, 3)
        positions_angstrom, cell_angstrom = fractional_to_cartesian(fractional, cell)

        label_value = getattr(row, label_column) if label_column in dataframe.columns else None
        weight_value = getattr(row, weight_column) if (weight_column is not None and weight_column in dataframe.columns) else 1.0

        config = gdata.atomic.Configuration(
            atomic_numbers=np.array(ALANINE_ATOMIC_NUMBERS, dtype=int),
            positions=positions_angstrom,
            cell=cell_angstrom,
            pbc=(True, True, True),
            node_labels=None,
            graph_labels=None if label_value is None else np.array([[label_value]], dtype=float),
            weight=weight_value,
        )
        configurations.append(config)

    z_table = gdata.atomic.AtomicNumberTable.from_zs(ALANINE_ATOMIC_NUMBERS)
    dataset = gdata.create_dataset_from_configurations(
        configurations,
        z_table=z_table,
        cutoff=cutoff_angstrom,
        buffer=0.0,
        cutoff_l=-1.0,
        remove_isolated_nodes=False,
        show_progress=show_progress,
    )

    bias_values = dataframe.get('opes.bias', pd.Series(np.zeros(len(dataframe))))                   + dataframe.get('bias', pd.Series(np.zeros(len(dataframe))))
    bias_tensor = torch.tensor(bias_values.values, dtype=torch.get_default_dtype())

    return dataset, bias_tensor



> **Note**
> The helper functions deliberately create the graphs directly from the COLVAR files in order to keep this tutorial self-contained. When working with molecular dynamics trajectories, you can instead rely on `mlcolvar.graph.utils.io.create_dataset_from_trajectories` to build the dataset from topology and trajectory files.



## 3. Iteration 0 – training on unbiased basin data

We start by learning a committor using only unbiased trajectories sampled from the two metastable basins (state A and state B). This iteration imposes the boundary conditions and produces an initial classifier-like model.

The dataset is downloaded directly from the public repository used in the example notebook. We subsample the unbiased trajectories by taking one frame every five time steps to keep the training set manageable.


In [ ]:
# URLs for the unbiased simulations (state A and state B)
unbiased_filenames = [
    'https://raw.githubusercontent.com/EnricoTrizio/committor_2.0/refs/heads/main/alanine/unbiased_sims/COLVAR_A',
    'https://raw.githubusercontent.com/EnricoTrizio/committor_2.0/refs/heads/main/alanine/unbiased_sims/COLVAR_B',
]

unbiased_load_args = [
    {'start': 0, 'stop': 10000, 'stride': 5},
    {'start': 0, 'stop': 10000, 'stride': 5},
]

unbiased_df = load_committor_dataframe(unbiased_filenames, unbiased_load_args)
print('Loaded dataframe shape:', unbiased_df.shape)

CUTOFF_ANGSTROM = 6.0
unbiased_dataset, unbiased_bias = dataframe_to_graph_dataset(
    unbiased_df,
    cutoff_angstrom=CUTOFF_ANGSTROM,
    label_column='label',
    weight_column=None,
    show_progress=True,
)
print(unbiased_dataset)



### 3.1 Assigning training weights

The Kolmogorov variational loss requires per-sample weights that account for the statistical bias of the input data. In the first iteration the trajectories are unbiased, therefore the bias is identically zero and all configurations receive the same weight. We still go through the `compute_committor_weights` helper so that the dataset is updated consistently.


In [ ]:
dummy_bias = torch.zeros(len(unbiased_dataset), dtype=torch.get_default_dtype())

unbiased_dataset = gnn_utils.compute_committor_weights(
    dataset=unbiased_dataset,
    bias=dummy_bias,
    beta=1.0 / (0.0083144621 * 300.0),
)

BATCH_SIZE = 256
datamodule_iter0 = gdata.GraphDataModule(
    unbiased_dataset,
    lengths=(1.0,),
    batch_size=BATCH_SIZE,
    random_split=False,
    shuffle=False,
)
datamodule_iter0.setup()
print(datamodule_iter0)



### 3.2 Define and train the GNN committor

We instantiate a `GraphCommittor` using a Geometric Vector Perceptron (GVP) backbone. The constructor needs the list of unique atomic numbers (available as `dataset.atomic_numbers`) and the corresponding atomic masses for the kinetic term of the Kolmogorov loss. The loss hyper-parameters are kept close to the defaults used in the feed-forward tutorial. The model returns both the raw network output `z` and the sigmoid-transformed committor `q`.

Training is handled by PyTorch Lightning. The following cell configures a trainer that runs for 80 epochs, logs the loss components, and prints a summary when running in Colab.


In [ ]:
type_masses = gdata.atomic.get_masses(unbiased_dataset.atomic_numbers)

model_iter0 = GraphCommittor(
    cutoff=CUTOFF_ANGSTROM,
    atomic_numbers=unbiased_dataset.atomic_numbers,
    atomic_masses=type_masses,
    model_name='GVPModel',
    model_options={
        'n_layers': 4,
        'n_hidden_channels': 128,
        'n_mlp_layers': 2,
        'n_mlp_channels': 256,
        'dropout': 0.0,
    },
    extra_loss_options={
        'alpha': 1.0,
        'gamma': 100.0,
        'delta_f': 0.0,
        'sigmoid_p': 3.0,
        'penalty_weight': 10.0,
        'z_threshold': 10.0,
        'n_bootstrap': 5,
    },
    optimizer_options={
        'optimizer': {'lr': 1e-3, 'weight_decay': 1e-5},
        'lr_scheduler': {
            'scheduler': torch.optim.lr_scheduler.ExponentialLR,
            'gamma': 0.9995,
        },
    },
)

trainer_iter0 = lightning.Trainer(
    max_epochs=80,
    accelerator='auto',
    devices='auto',
    log_every_n_steps=10,
    enable_progress_bar=True,
)

trainer_iter0.fit(model_iter0, datamodule=datamodule_iter0)



### 3.3 Inspect the learned committor

After training we can evaluate the model on the full dataset and visualise the predicted committor on the Ramachandran (`\phi`, `\psi`) plane. The plots compare the raw network output `z` and the sigmoid-transformed committor `q`. The scatter plot uses the same small marker size as the example notebook to emphasize the density of visited configurations.


In [ ]:
from torch_geometric.data import Batch

batch_iter0 = Batch.from_data_list(unbiased_dataset)
model_iter0.eval()
with torch.no_grad():
    zq_iter0 = model_iter0(batch_iter0.to_dict())

z_values = zq_iter0[:, 0].cpu().numpy()
q_values = zq_iter0[:, 1].cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(unbiased_df['phi'], unbiased_df['psi'], c=z_values, s=4, cmap='fessa')
axes[0].set_title('$z$ output (iteration 0)')
axes[0].set_xlabel('$\phi$ [rad]')
axes[0].set_ylabel('$\psi$ [rad]')

axes[1].scatter(unbiased_df['phi'], unbiased_df['psi'], c=q_values, s=4, cmap='fessa')
axes[1].set_title('$q$ committor (iteration 0)')
axes[1].set_xlabel('$\phi$ [rad]')
axes[1].set_ylabel('$\psi$ [rad]')

plt.tight_layout()
plt.show()



## 4. Iteration 1 – including biased trajectories

The initial classifier provides the correct boundary behaviour but underestimates the transition region because it does not see barrier-crossing configurations. To refine the committor we add the first round of OPES-biased simulations (one trajectory started from each basin). The biased trajectories are reweighted using the Kolmogorov bias computed from the previous model.

In practice, the workflow for additional iterations is:

1. Load the new data and append it to the dataset.
2. Evaluate the current committor to obtain the Kolmogorov bias.
3. Compute the new training weights using the stored bias values from the simulations.
4. Retrain (or continue training) the model on the augmented dataset.

The next cells implement these steps for iteration 1.


In [ ]:
biased_filenames = [
    'https://raw.githubusercontent.com/EnricoTrizio/committor_2.0/refs/heads/main/alanine/biased_sims/iter_0/A/COLVAR',
    'https://raw.githubusercontent.com/EnricoTrizio/committor_2.0/refs/heads/main/alanine/biased_sims/iter_0/B/COLVAR',
]

biased_load_args = [
    {'start': 1000, 'stop': 10000, 'stride': 1},
    {'start': 1000, 'stop': 10000, 'stride': 1},
]

biased_df = load_committor_dataframe(biased_filenames, biased_load_args, label_offset=len(unbiased_filenames))
print('Biased dataframe shape:', biased_df.shape)

combined_df = pd.concat([unbiased_df, biased_df], ignore_index=True)
combined_dataset, combined_bias_raw = dataframe_to_graph_dataset(
    combined_df,
    cutoff_angstrom=CUTOFF_ANGSTROM,
    label_column='label',
    weight_column=None,
    show_progress=True,
)
print(combined_dataset)



### 4.1 Kolmogorov bias and updated weights

To reweight the biased simulations we evaluate the committor obtained in iteration 0 and compute the Kolmogorov bias on the entire dataset. The mass-weighted version of the bias generally yields more stable statistics for molecular systems, and we reuse the `get_dataset_kolmogorov_bias` helper for that purpose.

The per-configuration bias stored in the COLVAR files (columns `opes.bias` and `bias`) is summed with the Kolmogorov bias to obtain the total bias acting on each configuration. The resulting vector is then passed to `compute_committor_weights` to update the dataset in-place.


In [ ]:
kolmogorov_bias = gnn_utils.get_dataset_kolmogorov_bias(
    model=model_iter0,
    dataset=combined_dataset,
    beta=1.0 / (0.0083144621 * 300.0),
    epsilon=1e-6,
    lambd=2.0,
    weighted=True,
    batch_size=BATCH_SIZE,
    show_progress=True,
)

combined_df = combined_df.copy()
combined_df['total_bias'] = combined_df['opes.bias'] + combined_df['bias'] + kolmogorov_bias.squeeze()
combined_bias_tensor = torch.tensor(combined_df['total_bias'].values, dtype=torch.get_default_dtype())

combined_dataset = gnn_utils.compute_committor_weights(
    dataset=combined_dataset,
    bias=combined_bias_tensor,
    beta=1.0 / (0.0083144621 * 300.0),
)

datamodule_iter1 = gdata.GraphDataModule(
    combined_dataset,
    lengths=(1.0,),
    batch_size=BATCH_SIZE,
    random_split=False,
    shuffle=True,
)
datamodule_iter1.setup()
print(datamodule_iter1)



### 4.2 Continue training on the augmented dataset

To keep the tutorial simple we reuse the model instance from iteration 0 and continue training on the expanded dataset. In a production workflow you may decide to reinitialise the model or adapt the learning rate schedule based on the loss trends.


In [ ]:
trainer_iter1 = lightning.Trainer(
    max_epochs=120,
    accelerator='auto',
    devices='auto',
    log_every_n_steps=10,
    enable_progress_bar=True,
)

trainer_iter1.fit(model_iter0, datamodule=datamodule_iter1)



### 4.3 Visualise the refined committor and the sampling

The refined committor should provide a much smoother description of the transition region. We also look at the distribution of sampled configurations in the biased simulations, coloured by the OPES bias value to highlight the enhanced exploration of the barrier region.


In [ ]:
from torch_geometric.data import Batch

batch_iter1 = Batch.from_data_list(combined_dataset)
model_iter0.eval()
with torch.no_grad():
    zq_iter1 = model_iter0(batch_iter1.to_dict())

z_values_1 = zq_iter1[:, 0].cpu().numpy()
q_values_1 = zq_iter1[:, 1].cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(combined_df['phi'], combined_df['psi'], c=z_values_1, s=3, cmap='fessa')
axes[0].set_title('$z$ output (iteration 1)')
axes[0].set_xlabel('$\phi$ [rad]')
axes[0].set_ylabel('$\psi$ [rad]')

axes[1].scatter(combined_df['phi'], combined_df['psi'], c=q_values_1, s=3, cmap='fessa')
axes[1].set_title('$q$ committor (iteration 1)')
axes[1].set_xlabel('$\phi$ [rad]')
axes[1].set_ylabel('$\psi$ [rad]')
plt.tight_layout()
plt.show()

# Visualise the sampling coloured by the OPES bias
df_biased_only = combined_df[combined_df['label'] >= len(unbiased_filenames)].copy()
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for walker_id, ax_col in enumerate(axes.T):
    subset = df_biased_only[df_biased_only['walker'] == walker_id]

    ax = ax_col[0]
    sc = ax.scatter(subset['phi'], subset['psi'], c=subset['opes.bias'], s=3, cmap='fessa')
    ax.set_title(f'Walker {walker_id}: $(\phi,\psi)$')
    ax.set_xlabel('$\phi$ [rad]')
    ax.set_ylabel('$\psi$ [rad]')
    plt.colorbar(sc, ax=ax, label='OPES bias [kJ/mol]')

    ax = ax_col[1]
    idx = subset.index.to_numpy()
    sc = ax.scatter(subset['time'] / 1000.0, q_values_1[idx], c=subset['opes.bias'], s=3, cmap='fessa')
    ax.set_title(f'Walker {walker_id}: committor vs time')
    ax.set_xlabel('Time [ns]')
    ax.set_ylabel('q')
    plt.colorbar(sc, ax=ax, label='OPES bias [kJ/mol]')

plt.tight_layout()
plt.show()



## 5. Exporting the model to TorchScript

Before coupling the trained committor with enhanced sampling codes it is convenient to export the model to TorchScript. The GNN committor stores both the raw `z` and the sigmoid `q` outputs. We typically export two versions of the model: one with the sigmoid disabled (for post-processing) and one with the sigmoid active (for on-the-fly evaluation inside PLUMED).


In [ ]:
model_iter0.preprocessing = None

model_iter0.sigmoid = None
model_iter0.to_torchscript('alanine_committor_iter1_z.pt', method='trace')

model_iter0.sigmoid = torch.nn.Sigmoid()
model_iter0.to_torchscript('alanine_committor_iter1_q.pt', method='trace')

print('Saved TorchScript checkpoints for iteration 1.')



## 6. Next steps

You can continue the iterative scheme by launching new OPES simulations driven by the refined committor, collecting the resulting trajectories, and repeating the reweighting and training procedure. The helper functions introduced in this tutorial can be easily adapted to automate the workflow for additional iterations or for different molecular systems.

**Key takeaways**

* The GNN module requires explicit atomic numbers and positions. The COLVAR files used in the feed-forward tutorial already contain the necessary information (scaled coordinates and cell vectors) to reconstruct the graphs.
* `mlcolvar.graph.cvs.committor.GraphCommittor` combines a GNN backbone with the Kolmogorov variational loss and boundary constraints, mirroring the functionality of the dense neural network committor.
* The Kolmogorov bias can be computed directly on graph datasets via `gnn_utils.get_dataset_kolmogorov_bias`, enabling a seamless iterative refinement loop with enhanced sampling data.

Happy exploring!
